# Popolazione per Comune per Anno

Legge i file `POSAS_<anno>_it_Comuni.csv` e costruisce una griglia:
- Righe: comuni
- Colonne: anni
- Valori: popolazione totale (riga con età=999, ultima colonna)

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
def parse_posas_file(filepath):
    """
    Legge un file POSAS_<anno>_it_Comuni.csv.
    Formato: separatore ';', valori quotati, prima riga = titolo da saltare.
    Restituisce un DataFrame con colonne: codice, comune, eta, totale
    """
    df = pd.read_csv(
        filepath,
        sep=';',
        skiprows=1,       # salta la riga del titolo
        dtype=str,        # leggi tutto come stringa per gestire celle vuote
        encoding='utf-8'
    )
    # Rinomina le colonne che ci servono (prime 3 + ultima)
    df.columns = [c.strip().strip('"') for c in df.columns]
    col_codice = df.columns[0]   # "Codice comune"
    col_comune = df.columns[1]   # "Comune"
    col_eta    = df.columns[2]   # "Età"
    col_totale = df.columns[-1]  # "Totale"

    result = pd.DataFrame({
        'codice': df[col_codice].str.strip('"'),
        'comune': df[col_comune].str.strip('"'),
        'eta':    pd.to_numeric(df[col_eta].str.strip('"'), errors='coerce'),
        'totale': pd.to_numeric(df[col_totale].str.strip('"'), errors='coerce'),
    })
    return result.dropna(subset=['eta', 'totale'])

In [ ]:
ANNI = list(range(2019, 2026))  # 2019-2025
DATA_DIR = Path('/content')

risultati = {}

for anno in ANNI:
    filepath = DATA_DIR / f'POSAS_{anno}_it_Comuni.csv'
    if not filepath.exists():
        print(f'File non trovato: {filepath}')
        continue
    df = parse_posas_file(filepath)
    totali = df[df['eta'] == 999][['codice', 'comune', 'totale']].copy()
    totali = totali.set_index(['codice', 'comune'])['totale']
    risultati[anno] = totali
    print(f'{anno}: {len(totali)} comuni caricati')

print(f'\nAnni caricati: {list(risultati.keys())}')

In [ ]:
# Costruisci la griglia: comuni x anni
griglia = pd.DataFrame(risultati)
griglia.index.names = ['codice', 'comune']
griglia.columns.name = 'anno'

print(f'Griglia: {griglia.shape[0]} comuni x {griglia.shape[1]} anni')
griglia.head(10)

In [ ]:
# Verifica: valori mancanti (comuni presenti solo in alcuni anni)
mancanti = griglia.isnull().sum()
if mancanti.sum() > 0:
    print('Valori mancanti per anno:')
    print(mancanti[mancanti > 0])
else:
    print('Nessun valore mancante')

In [ ]:
# Salva il risultato in CSV
output_path = DATA_DIR / 'popolazione_comuni_per_anno.csv'
griglia.to_csv(output_path)
print(f'Salvato in: {output_path}')

In [ ]:
# Esempio: cerca un comune specifico
comune_cerca = 'Abano Terme'
mask = griglia.index.get_level_values('comune') == comune_cerca
if mask.any():
    print(f'Popolazione di {comune_cerca} per anno:')
    print(griglia[mask].T)
else:
    print(f'Comune "{comune_cerca}" non trovato')